# Notebook 5 — Non-LLM prediction benchmarks: OLS and Lasso (exact LLM input match)

## Purpose

This notebook tests whether preservation of the normalized income–dispersion profile is a generic consequence of **point prediction**, rather than a distinctive property of large language models.

It trains OLS and Lasso on the **same realized respondent information that was serialized into the LLM prompts in Notebook 1**. To avoid any mismatch between the frozen numeric columns and the actual prompts, the benchmark predictors are reconstructed directly from `user_description`. For every mapped WVS item, the numeric response is extracted only when that item appeared in the respondent's LLM profile. Items omitted from the profile remain missing and are represented explicitly as missingness during statistical preprocessing. Age, sex, and country are taken from the same fields used to construct the profile opening sentence.

Both models generate strictly out-of-fold predictions for every respondent. The resulting predictions are then passed through the same raw-SD and normalized-profile analysis used for the LLMs.

This is a **post-preregistration benchmark analysis** and should not be described as confirmatory.


In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 20260926
N_OUTER_FOLDS = 5
N_LASSO_INNER_FOLDS = 3
N_BOOT = 200

# Primary benchmark uses continuous point predictions clipped to the valid Q49 range.
# Integer-rounded versions are also reported as a sensitivity check because the LLMs
# were constrained to return integer responses from 1 to 10.
CLIP_LOW, CLIP_HIGH = 1.0, 10.0

print("Configuration loaded.")


In [ ]:
# -------------------------
# Paths
# -------------------------

PROFILE_CANDIDATES = [
    Path("output/full_wvs/profiles_for_prediction.csv"),
    Path("../output/full_wvs/profiles_for_prediction.csv"),
    Path("profiles_for_prediction.csv"),
]

LLM_PROFILE_CANDIDATES = [
    Path("output/full_wvs/analysis_locked/03_primary_normalized_sd_profiles.csv"),
    Path("output/full_wvs/03_primary_normalized_sd_profiles.csv"),
    Path("analysis_locked/03_primary_normalized_sd_profiles.csv"),
    Path("03_primary_normalized_sd_profiles.csv"),
]

LLM_TEST_CANDIDATES = [
    Path("output/full_wvs/analysis_locked/04_PRIMARY_confirmatory_tests.csv"),
    Path("output/full_wvs/04_PRIMARY_confirmatory_tests.csv"),
    Path("analysis_locked/04_PRIMARY_confirmatory_tests.csv"),
    Path("04_PRIMARY_confirmatory_tests.csv"),
]

OUTDIR = Path("output/full_wvs/non_llm_benchmarks_exact_inputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

PROFILE_PATH = first_existing(PROFILE_CANDIDATES)
if PROFILE_PATH is None:
    raise FileNotFoundError(
        "Could not find profiles_for_prediction.csv. Run Notebook 1 first, "
        "or edit PROFILE_CANDIDATES above."
    )

LLM_PROFILE_PATH = first_existing(LLM_PROFILE_CANDIDATES)
LLM_TEST_PATH = first_existing(LLM_TEST_CANDIDATES)

print("Profiles:", PROFILE_PATH)
print("LLM normalized profiles:", LLM_PROFILE_PATH if LLM_PROFILE_PATH else "not found (optional)")
print("LLM confirmatory summary:", LLM_TEST_PATH if LLM_TEST_PATH else "not found (optional)")
print("Output directory:", OUTDIR)


In [ ]:
# -------------------------
# Load the exact Notebook 1 analysis sample
# -------------------------

df = pd.read_csv(PROFILE_PATH, low_memory=False)

required = ["WVS_ROW_ID", "Q49", "Q288", "COUNTRY_NAME"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Re-enforce the preregistered eligibility rule as a safeguard.
df = df.loc[df["Q49"].between(1, 10) & df["Q288"].between(1, 10)].copy()
df = df.sort_values("WVS_ROW_ID").reset_index(drop=True)

if len(df) != 93901:
    print(f"WARNING: N={len(df):,}; expected 93,901 in the full analysis sample.")

print(f"Analysis N: {len(df):,}")
print(f"Countries/territories: {df['COUNTRY_NAME'].nunique():,}")
print(df[["Q49", "Q288"]].describe())


## Exact reconstruction of the LLM information set

Notebook 1 created each `user_description` by looping over `QUESTION_MAPPING` and including a survey item only when that respondent had a nonmissing value. The benchmark therefore reconstructs those realized inputs directly from the serialized profile text.

This avoids relying on frozen numeric columns that may contain missing values even when the answer was present in the prompt. It also preserves prompt-level missingness: an omitted item is coded as missing rather than imputed as though the LLM had observed it.

Question wording is constant across respondents and therefore contains no respondent-level predictive information beyond identifying the item. Verbal answer labels are deterministic functions of the numeric response codes, so dummy-coding the recovered response codes supplies the same respondent-level information.


In [ ]:
# Exact Notebook 1 mapping used to construct user_description
QUESTION_MAPPING = {'Q1': "Indicate how important is 'Family' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)", 'Q2': "Indicate how important is 'Friends' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)", 'Q3': "Indicate how important is 'Leisure time' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)", 'Q4': "Indicate how important is 'Politics' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)", 'Q5': "Indicate how important is 'Work' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)", 'Q6': "Indicate how important is 'Religion' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)", 'Q47': 'All in all, how would you describe your state of health these days? (1-Very good, 2-Good, 3-Fair, 4-Poor, 5-Very poor)', 'Q48': 'How much freedom of choice and control do you feel you have over the way your life turns out? (1-No choice at all, 10-A great deal of choice)', 'Q51': 'In the last 12 months, how often have you or your family gone without enough food to eat? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)', 'Q52': 'In the last 12 months, how often have you or your family felt unsafe from crime in your home? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)', 'Q53': 'In the last 12 months, how often have you or your family gone without medicine or medical treatment that you needed? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)', 'Q54': 'In the last 12 months, how often have you or your family gone without a cash income? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)', 'Q55': 'In the last 12 months, how often have you or your family gone without a safe shelter over your head? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)', 'Q56': "Comparing your standard of living with your parents' standard of living when they were about your age (1-Better off, 2-Worse off, 3-About the same)", 'Q57': 'Generally speaking, would you say that most people can be trusted or that you need to be very careful in dealing with people? (1-Most people can be trusted, 2-Need to be very careful)', 'Q66': 'How much confidence do you have in the press? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)', 'Q69': 'How much confidence do you have in the police? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)', 'Q71': 'How much confidence do you have in the government? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)', 'Q72': 'How much confidence do you have in political parties? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)', 'Q75': 'How much confidence do you have in universities? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)', 'Q76': 'How much confidence do you have in elections? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)', 'Q106': 'Views on income equality (1-Incomes should be made more equal, 10-There should be greater incentives for individual effort)', 'Q112': 'Views on corruption in your country (1-There is no corruption in my country, 10-There is abundant corruption in my country)', 'Q131': 'How secure do you feel these days in your neighborhood? (1-Very secure, 2-Quite secure, 3-Not very secure, 4-Not at all secure)', 'Q164': 'How important is God in your life? (1-Not at all important, 10-Very important)', 'Q171': 'How often do you attend religious services? (1-More than once a week, 2-Once a week, 3-Once a month, 4-Only on special holy days, 5-Once a year, 6-Less often, 7-Never/practically never)', 'Q173': 'Would you say you are (1-A religious person, 2-Not a religious person, 3-An atheist)', 'Q250': 'How important is it for you to live in a country that is governed democratically? (1-Not at all important, 10-Absolutely important)', 'Q254': "How proud are you to be [country's nationality]? (1-Very proud, 2-Quite proud, 3-Not very proud, 4-Not at all proud, 5-I am not)", 'Q260': "Respondent's sex (1-Male, 2-Female)", 'Q262': 'Age', 'Q263': 'Were you born in this country or are you an immigrant to this country? (1-Born in this country, 2-Immigrant to this country)', 'Q269': 'Are you a citizen of this country? (1-Yes, 2-No)', 'Q270': 'How many people, including children, live here regularly as members of this household?', 'Q273': 'Current marital status (1-Married, 2-Living together as married, 3-Divorced, 4-Separated, 5-Widowed, 6-Single)', 'Q274': 'Number of children', 'Q275': 'Highest educational level attained (0-Early childhood/no education to 8-Doctoral or equivalent)', 'Q279': 'Employment status (1-Full time, 2-Part time, 3-Self employed, 4-Retired/pensioned, 5-Housewife, 6-Student, 7-Unemployed, 8-Other)', 'Q281': 'Occupational group (0-Never had a job to 10-Farm proprietor/manager)', 'Q284': 'Sector of employment (1-Government/public institution, 2-Private business/industry, 3-Private non-profit organization)', 'Q285': 'Chief wage earner in household (1-Yes, 2-No)', 'Q287': 'Subjective social class (1-Upper class, 2-Upper middle class, 3-Lower middle class, 4-Working class, 5-Lower class)', 'Q288': 'Household income group in your country, counting wages, salaries, pensions and other household income (1-Lowest income group, 10-Highest income group)', 'Q289': 'Religion/denomination (0-None, 1-Roman Catholic, 2-Protestant, 3-Orthodox, 4-Jewish, 5-Muslim, 6-Hindu, 7-Buddhist, 8-Other Christian, 9-Other)', 'E1': "Respondent's literacy (1-Literate, 2-Illiterate)", 'H': 'Settlement type where the interview was conducted (1-Capital city, 2-Regional center, 3-District center, 4-Another city/town, 5-Village)'}

if "user_description" not in df.columns:
    raise ValueError("profiles_for_prediction.csv must contain user_description for exact input reconstruction.")

# Start from the profile text that was actually sent to the LLMs.
profile_text = df["user_description"].fillna("").astype(str)

def extract_prompt_value(text_series, wording):
    # Robust parser: avoid pandas .str chaining, which can break when an
    # intermediate Series is inferred as non-string in some pandas versions.
    # Notebook 1 serialized each observed item as:
    # the person answered the question '<wording>' as '<numeric> - <label>'
    marker = "the person answered the question '" + wording + "' as '"

    def parse_one(x):
        if pd.isna(x):
            return np.nan, False
        txt = str(x)
        pos = txt.find(marker)
        if pos < 0:
            return np.nan, False
        start = pos + len(marker)
        tail = txt[start:]
        raw = tail.split(" - ", 1)[0].strip().strip("'\"")
        try:
            return float(raw), True
        except (TypeError, ValueError):
            return np.nan, True

    parsed = [parse_one(x) for x in text_series.tolist()]
    out = pd.Series([v for v, _ in parsed], index=text_series.index, dtype=float)
    has_item = pd.Series([a for _, a in parsed], index=text_series.index, dtype=bool)
    return out, has_item

# Reconstruct every item that was serialized in the loop. Q260/Q262 were supplied
# in the profile opening sentence instead of repeated in the loop.
prompt_df = pd.DataFrame(index=df.index)
audit_rows = []

for q, wording in QUESTION_MAPPING.items():
    if q in {"Q260", "Q262"}:
        continue
    vals, appeared = extract_prompt_value(profile_text, wording)
    prompt_df[q] = vals
    n_appeared = int(appeared.sum())
    n_parsed = int(vals.notna().sum())
    audit_rows.append({
        "variable": q,
        "source": "parsed_from_user_description",
        "n_appeared_in_prompt": n_appeared,
        "n_parsed_numeric": n_parsed,
        "parse_match": n_appeared == n_parsed,
    })

# Same fields used in the opening sentence of Notebook 1.
prompt_df["Q260"] = pd.to_numeric(df["Q260"], errors="coerce")
prompt_df["Q262"] = pd.to_numeric(df["Q262"], errors="coerce")
prompt_df["COUNTRY_NAME"] = df["COUNTRY_NAME"].astype("string")

for q in ["Q260", "Q262"]:
    audit_rows.append({
        "variable": q,
        "source": "profile_opening_field",
        "n_appeared_in_prompt": int(prompt_df[q].notna().sum()),
        "n_parsed_numeric": int(prompt_df[q].notna().sum()),
        "parse_match": True,
    })
audit_rows.append({
    "variable": "COUNTRY_NAME",
    "source": "profile_opening_field",
    "n_appeared_in_prompt": int(prompt_df["COUNTRY_NAME"].notna().sum()),
    "n_parsed_numeric": np.nan,
    "parse_match": True,
})

predictor_audit = pd.DataFrame(audit_rows)
predictor_audit.to_csv(OUTDIR / "00_predictor_audit.csv", index=False)

# Strict QA: every serialized numeric response must be recovered.
bad_parse = predictor_audit.loc[predictor_audit["parse_match"].eq(False)]
if len(bad_parse):
    display(bad_parse)
    raise AssertionError("At least one prompt variable could not be parsed exactly from user_description.")

# Q288 must be present in every eligible profile by Notebook 1 QA.
if not prompt_df["Q288"].notna().all():
    raise AssertionError("Exact reconstruction failed: at least one eligible profile is missing parsed Q288.")

PROFILE_VARS = list(QUESTION_MAPPING.keys()) + ["COUNTRY_NAME"]
PREDICTORS = [c for c in PROFILE_VARS if c in prompt_df.columns]

# Continuous quantities are treated as continuous; discrete WVS responses are dummy-coded.
CONTINUOUS = [c for c in ["Q262", "Q270", "Q274"] if c in PREDICTORS]
CATEGORICAL = [c for c in PREDICTORS if c not in CONTINUOUS]

assert "Q49" not in PREDICTORS
assert "Q46" not in PREDICTORS

print(f"Exact prompt predictors reconstructed: {len(PREDICTORS)}")
print(f"Continuous: {CONTINUOUS}")
print(f"Categorical/dummy-coded: {len(CATEGORICAL)}")
print("\nPrompt reconstruction audit:")
display(predictor_audit)

# Useful comparison against frozen columns: these are variables recovered from the prompt
# even if the corresponding numeric column is empty in profiles_for_prediction.csv.
recovered_despite_empty_column = []
for q in QUESTION_MAPPING:
    if q in {"Q260", "Q262"}:
        continue
    frozen_nonmissing = int(df[q].notna().sum()) if q in df.columns else 0
    parsed_nonmissing = int(prompt_df[q].notna().sum())
    if frozen_nonmissing == 0 and parsed_nonmissing > 0:
        recovered_despite_empty_column.append((q, parsed_nonmissing))

print("Recovered from actual prompt text despite empty/missing frozen numeric column:")
print(recovered_despite_empty_column)


In [ ]:
# -------------------------
# Preprocessing
# -------------------------

def make_ohe():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=True,
            dtype=np.float64,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse=True,
            dtype=np.float64,
        )

# Missingness is part of the realized LLM information set because a missing item was omitted
# from user_description. Preserve that information explicitly rather than replacing it silently.
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-9999)),
    ("onehot", make_ohe()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, CONTINUOUS),
        ("cat", categorical_pipe, CATEGORICAL),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

ols_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", LinearRegression()),
])

lasso_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", LassoCV(
        cv=N_LASSO_INNER_FOLDS,
        alphas=np.logspace(-4, 1, 60),
        max_iter=10000,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        selection="cyclic",
    )),
])

print("Pipelines ready. Missing prompt items are represented explicitly.")


## Five-fold cross-fitting

Folds are stratified by country so that each held-out fold preserves the cross-national composition of the full sample. Every prediction is generated by a model that was estimated without that respondent's `Q49`.

Lasso chooses its penalty **inside each outer training fold** using three-fold cross-validation. This avoids using held-out outcomes to tune the penalty.


In [ ]:
# -------------------------
# Strictly out-of-fold predictions
# -------------------------

X = prompt_df[PREDICTORS].copy()
y = df["Q49"].astype(float).to_numpy()
strata = df["COUNTRY_NAME"].astype(str).to_numpy()

outer = StratifiedKFold(
    n_splits=N_OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

pred_ols = np.full(len(df), np.nan, dtype=float)
pred_lasso = np.full(len(df), np.nan, dtype=float)
fold_id = np.full(len(df), -1, dtype=int)
fold_rows = []

for fold, (train_idx, test_idx) in enumerate(outer.split(X, strata), start=1):
    print(f"Outer fold {fold}/{N_OUTER_FOLDS}: train={len(train_idx):,}, test={len(test_idx):,}")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    ols = clone(ols_pipe)
    ols.fit(X_train, y_train)
    p_ols = ols.predict(X_test)
    pred_ols[test_idx] = p_ols

    lasso = clone(lasso_pipe)
    lasso.fit(X_train, y_train)
    p_lasso = lasso.predict(X_test)
    pred_lasso[test_idx] = p_lasso

    alpha = float(lasso.named_steps["model"].alpha_)
    fold_id[test_idx] = fold

    fold_rows.append({
        "fold": fold,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "lasso_alpha": alpha,
        "ols_mae_unclipped": mean_absolute_error(y_test, p_ols),
        "lasso_mae_unclipped": mean_absolute_error(y_test, p_lasso),
    })

assert np.isfinite(pred_ols).all()
assert np.isfinite(pred_lasso).all()
assert (fold_id > 0).all()

fold_audit = pd.DataFrame(fold_rows)
fold_audit.to_csv(OUTDIR / "01_crossfit_fold_audit.csv", index=False)
display(fold_audit)
print("Cross-fitting complete.")


In [ ]:
# -------------------------
# Construct bounded benchmark predictions
# -------------------------

bench = df[["WVS_ROW_ID", "COUNTRY_NAME", "B_COUNTRY_ALPHA", "Q49", "Q288"]].copy()
bench["fold"] = fold_id

bench["OLS_unclipped"] = pred_ols
bench["Lasso_unclipped"] = pred_lasso

bench["OLS"] = np.clip(pred_ols, CLIP_LOW, CLIP_HIGH)
bench["Lasso"] = np.clip(pred_lasso, CLIP_LOW, CLIP_HIGH)

# Sensitivity that exactly matches the integer response support imposed on the LLMs.
bench["OLS_integer"] = np.rint(bench["OLS"]).clip(CLIP_LOW, CLIP_HIGH).astype(int)
bench["Lasso_integer"] = np.rint(bench["Lasso"]).clip(CLIP_LOW, CLIP_HIGH).astype(int)

bench.to_csv(OUTDIR / "02_OOF_predictions_OLS_Lasso.csv", index=False)

print("Prediction ranges:")
for c in ["OLS_unclipped", "Lasso_unclipped", "OLS", "Lasso", "OLS_integer", "Lasso_integer"]:
    print(f"  {c:16s}: {bench[c].min():.3f} to {bench[c].max():.3f}")


In [ ]:
# -------------------------
# Predictive-performance diagnostics
# -------------------------

def perf_row(name, pred):
    pred = np.asarray(pred, dtype=float)
    return {
        "source": name,
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "R2": r2_score(y, pred),
        "prediction_mean": float(np.mean(pred)),
        "prediction_sd": float(np.std(pred, ddof=1)),
        "human_sd": float(np.std(y, ddof=1)),
        "overall_sd_ratio_vs_human": float(np.std(pred, ddof=1) / np.std(y, ddof=1)),
    }

performance = pd.DataFrame([
    perf_row("OLS", bench["OLS"]),
    perf_row("Lasso", bench["Lasso"]),
    perf_row("OLS_integer", bench["OLS_integer"]),
    perf_row("Lasso_integer", bench["Lasso_integer"]),
])
performance.to_csv(OUTDIR / "03_predictive_performance.csv", index=False)
display(performance)


## Income-group dispersion and normalization

The normalization exactly follows the main analysis. Human income-group shares are used as fixed weights for **every source**. Thus, differences in the normalized profile cannot arise from giving OLS or Lasso different income-group weights.


In [ ]:
# -------------------------
# Human weights and source-specific normalized profiles
# -------------------------

human_weights = (
    bench["Q288"].value_counts(normalize=True).sort_index().reindex(range(1, 11))
)
assert human_weights.notna().all()
assert np.isclose(human_weights.sum(), 1.0)

SOURCE_COLS = {
    "Human": "Q49",
    "OLS": "OLS",
    "Lasso": "Lasso",
    "OLS integer": "OLS_integer",
    "Lasso integer": "Lasso_integer",
}

def profile_for_source(data, source, col, weights=human_weights):
    g = (
        data.groupby("Q288", observed=True)[col]
        .agg(n="size", mean="mean", sd="std")
        .reindex(range(1, 11))
        .reset_index()
    )
    if g["sd"].isna().any():
        raise ValueError(f"Missing income-group SD for {source}")
    scale = float(np.sum(weights.values * g["sd"].values))
    g["source"] = source
    g["scale"] = scale
    g["normalized_sd"] = g["sd"] / scale
    return g[["source", "Q288", "n", "mean", "sd", "scale", "normalized_sd"]]

profiles = pd.concat(
    [profile_for_source(bench, s, c) for s, c in SOURCE_COLS.items()],
    ignore_index=True,
)
profiles.to_csv(OUTDIR / "04_benchmark_income_profiles.csv", index=False)
display(profiles)


In [ ]:
# -------------------------
# Structural-fidelity summaries
# -------------------------

human_prof = (
    profiles.loc[profiles["source"].eq("Human")]
    .sort_values("Q288")
    .set_index("Q288")["normalized_sd"]
)

x = np.arange(1, 11, dtype=float)
human_slope = float(np.polyfit(x, human_prof.values, 1)[0])

summary_rows = []
for source in ["OLS", "Lasso", "OLS integer", "Lasso integer"]:
    p = (
        profiles.loc[profiles["source"].eq(source)]
        .sort_values("Q288")
        .set_index("Q288")["normalized_sd"]
    )
    slope = float(np.polyfit(x, p.values, 1)[0])
    diff = p.values - human_prof.values
    summary_rows.append({
        "source": source,
        "human_normalized_slope": human_slope,
        "model_normalized_slope": slope,
        "slope_difference_model_minus_human": slope - human_slope,
        "profile_RMSD_vs_human": float(np.sqrt(np.mean(diff**2))),
        "profile_MAE_vs_human": float(np.mean(np.abs(diff))),
        "profile_Pearson_r_vs_human": float(np.corrcoef(p.values, human_prof.values)[0, 1]),
        "raw_dispersion_scale": float(profiles.loc[profiles["source"].eq(source), "scale"].iloc[0]),
        "human_raw_dispersion_scale": float(profiles.loc[profiles["source"].eq("Human"), "scale"].iloc[0]),
    })

structural = pd.DataFrame(summary_rows)
structural["scale_ratio_vs_human"] = structural["raw_dispersion_scale"] / structural["human_raw_dispersion_scale"]
structural.to_csv(OUTDIR / "05_benchmark_structural_fidelity.csv", index=False)
display(structural)


## Paired country-cluster bootstrap

The predictions themselves remain fixed. The bootstrap asks how much the **estimated human-versus-benchmark profile difference** varies with the sampled set of countries, using the same paired country resampling principle as the manuscript's primary profile inference.

This is intentionally not a bootstrap that re-trains OLS/Lasso on every draw; cross-fitting already protects the prediction benchmark from in-sample overfit, and re-training would change the estimand while adding substantial computation.


In [ ]:
# -------------------------
# Country-cluster bootstrap for slope differences
# -------------------------

rng = np.random.default_rng(RANDOM_STATE)
countries = bench["COUNTRY_NAME"].dropna().unique()
country_parts = {c: bench.loc[bench["COUNTRY_NAME"].eq(c)].copy() for c in countries}

BOOT_SOURCES = {"OLS": "OLS", "Lasso": "Lasso"}
boot_rows = []

for b in range(N_BOOT):
    draw = rng.choice(countries, size=len(countries), replace=True)
    # Keep duplicate sampled countries as distinct clusters by concatenating their respondent rows.
    db = pd.concat([country_parts[c] for c in draw], ignore_index=True)

    hp = profile_for_source(db, "Human", "Q49", weights=human_weights)
    hvals = hp.sort_values("Q288")["normalized_sd"].to_numpy()
    hslope = float(np.polyfit(x, hvals, 1)[0])

    for source, col in BOOT_SOURCES.items():
        mp = profile_for_source(db, source, col, weights=human_weights)
        mvals = mp.sort_values("Q288")["normalized_sd"].to_numpy()
        mslope = float(np.polyfit(x, mvals, 1)[0])
        boot_rows.append({
            "bootstrap": b + 1,
            "source": source,
            "human_slope": hslope,
            "model_slope": mslope,
            "slope_difference": mslope - hslope,
            "profile_RMSD_vs_human": float(np.sqrt(np.mean((mvals - hvals)**2))),
        })

    if (b + 1) % 100 == 0:
        print(f"Bootstrap {b+1}/{N_BOOT}")

boot = pd.DataFrame(boot_rows)
boot.to_csv(OUTDIR / "06_benchmark_country_bootstrap_replicates.csv", index=False)

ci_rows = []
for source, d in boot.groupby("source"):
    point = structural.loc[structural["source"].eq(source), "slope_difference_model_minus_human"].iloc[0]
    vals = d["slope_difference"].to_numpy()
    ci_rows.append({
        "source": source,
        "point_slope_difference": point,
        "bootstrap_SE": float(np.std(vals, ddof=1)),
        "CI95_low": float(np.quantile(vals, 0.025)),
        "CI95_high": float(np.quantile(vals, 0.975)),
        "p_two_sided": float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0))),
        "mean_bootstrap_RMSD": float(d["profile_RMSD_vs_human"].mean()),
    })

boot_summary = pd.DataFrame(ci_rows)
boot_summary.to_csv(OUTDIR / "07_benchmark_country_bootstrap_summary.csv", index=False)
display(boot_summary)


## Direct comparison with the six LLMs

If Notebook 3's `03_primary_normalized_sd_profiles.csv` and `04_PRIMARY_confirmatory_tests.csv` are available, this section appends OLS and Lasso to the existing LLM comparison. If those files are not present, the notebook still completes all benchmark analyses above.


In [ ]:
# -------------------------
# Optional merge with existing Human + six-LLM results
# -------------------------

comparison = None
if LLM_TEST_PATH is not None:
    llm_tests = pd.read_csv(LLM_TEST_PATH)
    keep = [
        "model", "human_linear_trend", "model_linear_trend",
        "trend_difference_model_minus_human", "profile_rmsd_vs_human"
    ]
    llm_tests = llm_tests[[c for c in keep if c in llm_tests.columns]].copy()
    llm_tests = llm_tests.rename(columns={
        "model": "source",
        "human_linear_trend": "human_normalized_slope",
        "model_linear_trend": "model_normalized_slope",
        "trend_difference_model_minus_human": "slope_difference_model_minus_human",
        "profile_rmsd_vs_human": "profile_RMSD_vs_human",
    })
    llm_tests["source_type"] = "LLM"

    bm = structural.loc[structural["source"].isin(["OLS", "Lasso"]), [
        "source", "human_normalized_slope", "model_normalized_slope",
        "slope_difference_model_minus_human", "profile_RMSD_vs_human"
    ]].copy()
    bm["source_type"] = "trained statistical benchmark"

    comparison = pd.concat([llm_tests, bm], ignore_index=True, sort=False)
    comparison.to_csv(OUTDIR / "08_LLM_vs_statistical_benchmark_summary.csv", index=False)
    display(comparison)
else:
    print("LLM confirmatory summary not found; skipping merged summary.")


In [ ]:
# -------------------------
# Figure: Human + OLS + Lasso normalized profiles
# -------------------------

fig, ax = plt.subplots(figsize=(8.2, 5.2))
for source, lw, marker in [
    ("Human", 2.8, "o"),
    ("OLS", 2.0, "s"),
    ("Lasso", 2.0, "^"),
]:
    d = profiles.loc[profiles["source"].eq(source)].sort_values("Q288")
    ax.plot(d["Q288"], d["normalized_sd"], marker=marker, linewidth=lw, label=source)

ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_xlabel("Household income group (1 = lowest, 10 = highest)")
ax.set_ylabel("Normalized SD of life satisfaction")
ax.set_xticks(range(1, 11))
ax.set_title("Human and non-LLM benchmark income–dispersion profiles")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTDIR / "fig_benchmark_normalized_profiles.pdf", bbox_inches="tight")
fig.savefig(OUTDIR / "fig_benchmark_normalized_profiles.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# -------------------------
# Optional figure: append OLS/Lasso to Human + six LLMs
# -------------------------

if LLM_PROFILE_PATH is not None:
    llm_prof = pd.read_csv(LLM_PROFILE_PATH)
    needed = {"source", "Q288", "normalized_sd"}
    if not needed.issubset(llm_prof.columns):
        print("LLM profile file found but has unexpected columns; skipping combined figure.")
    else:
        fig, ax = plt.subplots(figsize=(9.0, 5.6))

        # Existing Human + LLM profiles
        for source, d in llm_prof.groupby("source"):
            d = d.sort_values("Q288")
            if source == "Human":
                ax.plot(d["Q288"], d["normalized_sd"], linewidth=3.0, label=source)
            else:
                ax.plot(d["Q288"], d["normalized_sd"], linewidth=1.25, alpha=0.72, label=source)

        # Statistical benchmarks
        for source, marker in [("OLS", "s"), ("Lasso", "^")]:
            d = profiles.loc[profiles["source"].eq(source)].sort_values("Q288")
            ax.plot(
                d["Q288"], d["normalized_sd"],
                linewidth=2.2, linestyle="--", marker=marker,
                label=source
            )

        ax.axhline(1.0, linestyle=":", linewidth=1)
        ax.set_xlabel("Household income group (1 = lowest, 10 = highest)")
        ax.set_ylabel("Normalized SD of life satisfaction")
        ax.set_xticks(range(1, 11))
        ax.set_title("Normalized income–dispersion profiles: humans, LLMs, OLS, and Lasso")
        ax.legend(frameon=False, ncol=2, fontsize=8)
        fig.tight_layout()
        fig.savefig(OUTDIR / "fig_LLM_vs_statistical_benchmarks.pdf", bbox_inches="tight")
        fig.savefig(OUTDIR / "fig_LLM_vs_statistical_benchmarks.png", dpi=220, bbox_inches="tight")
        plt.show()
else:
    print("LLM normalized-profile file not found; skipping combined figure.")


## Reading the result

The most useful outputs for the manuscript decision are:

- `03_predictive_performance.csv`: how much OLS/Lasso compress overall prediction variance;
- `05_benchmark_structural_fidelity.csv`: normalized slope, slope difference, RMSD, and profile correlation relative to humans;
- `07_benchmark_country_bootstrap_summary.csv`: country-cluster bootstrap uncertainty for the OLS/Lasso slope differences;
- `08_LLM_vs_statistical_benchmark_summary.csv`: direct comparison with the six LLMs, when Notebook 3 outputs are available;
- `fig_LLM_vs_statistical_benchmarks.pdf`: visual comparison of all profiles.

### Possible interpretations

**A. OLS/Lasso compress variance but do not preserve the human normalized profile as well as the LLMs.**  
This is the strongest result for the current paper. It shows that structural fidelity is not simply a mechanical consequence of point prediction from the available covariates.

**B. OLS/Lasso preserve the normalized profile about as well as the LLMs.**  
The finding remains useful, but the interpretation broadens: scale compression and structural preservation can coexist in predictive systems generally. The distinctive LLM result is then that pretrained models recover this structure without being trained on `Q49` in this WVS sample.

**C. OLS/Lasso preserve the profile substantially better than the LLMs.**  
The paper should present the statistical benchmark as an upper/reference benchmark and avoid implying that structural fidelity is uniquely strong in LLMs.

Do not choose the framing until the out-of-fold results have been inspected.


In [ ]:
# -------------------------
# Final manifest
# -------------------------

manifest = {
    "analysis": "non-LLM OLS/Lasso benchmark with exact prompt-matched inputs",
    "sample_n": int(len(df)),
    "n_countries": int(df["COUNTRY_NAME"].nunique()),
    "outcome": "Q49 life satisfaction",
    "income": "Q288",
    "n_predictors": int(len(PREDICTORS)),
    "predictor_source": "reconstructed from realized user_description plus opening age/sex/country fields",
    "outer_folds": N_OUTER_FOLDS,
    "lasso_inner_folds": N_LASSO_INNER_FOLDS,
    "country_bootstrap_reps": N_BOOT,
    "random_state": RANDOM_STATE,
    "primary_benchmark_predictions": "continuous OOF predictions clipped to [1,10]",
    "integer_sensitivity": True,
    "post_preregistration": True,
}

with open(OUTDIR / "benchmark_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nDone. Outputs saved to:", OUTDIR)


In [ ]:
# -------------------------
# Bundle Notebook 6 outputs for download
# -------------------------
from pathlib import Path
import zipfile

# Use the notebook's actual output directory defined above.
# Include all analysis outputs produced by Notebook 6.
allowed_suffixes = {".csv", ".pdf", ".png", ".json"}
files_to_zip = sorted(
    p for p in OUTDIR.iterdir()
    if p.is_file() and p.suffix.lower() in allowed_suffixes
)

zip_path = Path("/datasets/_deepnote_work/Notebook6_nonLLM_benchmark_EXACT_INPUTS_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in files_to_zip:
        z.write(p, arcname=p.name)

print(f"Created: {zip_path}")
print(f"Included {len(files_to_zip)} files:\n")
for p in files_to_zip:
    print("✓", p.name)

if not files_to_zip:
    print("\nNo output files were found. Run the analysis cells above first.")


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=7aaa7215-b731-433d-9b62-8be4a70a4410' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>